In [4]:
import torch
import matplotlib.pyplot as plt
from relaxations import Zonotope_Net, Zonotope
import utils

# 1️⃣ Load pretrained MNIST model
net = utils.load_net_from_patch_attacks("examples/mnist_nets/7x200_best.pth")
net.eval()

# 2️⃣ Take one MNIST image
dataset = utils.load_dataset_selected_labels_only("mnist", labels=None, test_set=True)
x, y = dataset[0]
x = x.unsqueeze(0)  # add batch dim

print(f"True label: {y}, model prediction: {net(x).argmax().item()}")

# 3️⃣ Create a Zonotope network and initialize
epsilon = 0.05
znet = Zonotope_Net(net, relu_transformer="zonotope")
znet.initialize(x, epsilon)

# 4️⃣ Propagate up to the first layer (or first ReLU)
for i, layer in enumerate(net.layers):
    znet.apply_layer(i)
    if isinstance(layer, torch.nn.ReLU):
        print(f"Stopping after layer {i} ({layer})")
        break

# 5️⃣ Get resulting Zonotope and project to 2D
z = znet.relaxation_at_layers[-1]
a0 = z.a0[:, [0, 1]]
A = z.A[:, [0, 1]]
z2d = Zonotope(a0, A)

# 6️⃣ Plot
fig, ax = plt.subplots(figsize=(4,4))
ax.add_patch(z2d.plot(color='tab:blue'))
ax.set_aspect('equal', 'box')
plt.title("Zonotope after first ReLU layer (2D projection)")
plt.show()
